# Influencia y comunidades

Compara PageRank con el grado ponderado, y Louvain con Label Propagation. Ejecuta todas las celdas. Este notebook contiene su código y no usa los archivos de `test/`.


In [ ]:
RUN_MODE = "quick"  # "quick" mide 1K, 10K y 100K. "full" añade 1M y 10M.
SEED = 42
DATASET_FILE = ""  # Ruta a 2019-Oct.csv. Vacío: muestra sintética solo para validar.

print("Modo:", RUN_MODE)
print("Semilla:", SEED)
print("CSV:", DATASET_FILE or "sintético, solo validación")


In [ ]:
import platform
import time
import tracemalloc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "01_filtrado_probabilistico.ipynb").exists():
    ROOT = ROOT / "pruebas_algoritmos"

def sizes(mode):
    return [1_000, 10_000, 100_000] if mode == "quick" else [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

def repetitions(n, mode):
    if mode == "quick":
        return 7
    return 3 if n >= 1_000_000 else 100

def prepare_output(folder):
    results, figures = folder / "resultados", folder / "figuras"
    results.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return results, figures

def resolve_data(data):
    if not data:
        return None
    candidate = Path(data)
    options = [candidate, Path.cwd() / candidate, ROOT / candidate, ROOT.parent / candidate]
    found = next((option for option in options if option.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"No existe el CSV indicado: {data}")
    return found

def load_events(data, n, seed):
    found = resolve_data(data)
    if found:
        cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session", "brand", "category_code"]
        frame = pd.read_csv(found, nrows=n, usecols=lambda column: column in cols)
        return frame, "REES46 CSV"
    rng = np.random.default_rng(seed)
    products = max(300, min(n // 8, 20_000))
    users = max(200, min(n // 5, 50_000))
    frame = pd.DataFrame({
        "event_time": pd.date_range("2019-10-01", periods=n, freq="s"),
        "event_type": rng.choice(["view", "cart", "purchase"], size=n, p=[0.83, 0.12, 0.05]),
        "product_id": rng.integers(1_000_000, 1_000_000 + products, size=n),
        "price": np.round(rng.lognormal(4.2, 0.75, n), 2),
        "user_id": rng.integers(1, users, size=n),
        "user_session": rng.integers(1, max(20, n // 4), size=n).astype(str),
        "brand": rng.choice(["samsung", "apple", "sony", "unknown"], size=n),
        "category_code": rng.choice(["electronics.phone", "appliances.kitchen", "computers"], size=n),
    })
    return frame, "SINTÉTICO (solo validación; no usar en el informe)"

def measure(action, reps):
    timings = []
    for _ in range(reps):
        start = time.perf_counter()
        action()
        timings.append((time.perf_counter() - start) * 1_000)
    tracemalloc.start()
    action()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return float(np.mean(timings)), float(np.std(timings)), peak / 1024 ** 2

def record(rows, **fields):
    rows.append({
        "experimento": fields["experiment"], "algoritmo": fields["algorithm"], "n": fields["n"],
        "metrica": fields["metric"], "valor": fields["value"], "tiempo_promedio_ms": fields["time_ms"],
        "desviacion_ms": fields["std_ms"], "memoria_pico_mb": fields["memory_mb"],
        "repeticiones": fields["reps"], "semilla": fields["seed"],
        "plataforma": platform.platform(), "fuente": fields["source"],
    })

def save(rows, name, output):
    results, _ = prepare_output(output)
    path = results / f"{name}.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"Tabla: {path}")
    return path

def line_chart(frame, title, filename, output, value="tiempo_promedio_ms", log=True):
    _, figures = prepare_output(output)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for algorithm, group in frame.groupby("algoritmo"):
        group = group.sort_values("n")
        if log:
            group = group[group[value] > 0]
        if group.empty:
            continue
        ax.plot(group["n"], group[value], marker="o", label=algorithm)
    ax.set(title=title, xlabel="Elementos", ylabel=value.replace("_", " "))
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    path = figures / filename
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    print(f"Figura: {path}")
    return path

import random
from itertools import combinations

import networkx as nx

def graph_from(frame):
    graph = nx.Graph()
    buys = frame[frame["event_type"] == "purchase"]
    for _, group in buys.groupby("user_session"):
        products = list(map(int, group["product_id"].unique()))[:20]
        for a, b in combinations(products, 2):
            graph.add_edge(a, b, weight=graph.get_edge_data(a, b, {}).get("weight", 0) + 1)
    return graph


def pagerank(graph, damping=0.85, iterations=30):
    """Implementación sin SciPy, equivalente a la usada por el notebook v2."""
    nodes = list(graph)
    if not nodes:
        return {}
    scores = {node: 1 / len(nodes) for node in nodes}
    for _ in range(iterations):
        updated = {node: (1 - damping) / len(nodes) for node in nodes}
        for node in nodes:
            neighbors = list(graph[node].items())
            total = sum(data.get("weight", 1) for _, data in neighbors)
            if total == 0:
                for target in nodes:
                    updated[target] += damping * scores[node] / len(nodes)
            else:
                for target, data in neighbors:
                    updated[target] += damping * scores[node] * data.get("weight", 1) / total
        scores = updated
    return scores


def partition_agreement(first, second, seed):
    """Estabilidad: proporción de pares que conservan la co-pertenencia."""
    nodes = list(set().union(*first))
    if len(nodes) < 2:
        return 1.0
    labels = []
    for partition in (first, second):
        labels.append({node: index for index, group in enumerate(partition) for node in group})
    import random
    rng = random.Random(seed)
    pairs = [rng.sample(nodes, 2) for _ in range(min(2_000, len(nodes) * 2))]
    return sum((labels[0][a] == labels[0][b]) == (labels[1][a] == labels[1][b]) for a, b in pairs) / len(pairs)


def run(mode="quick", data=None, seed=42, out=None) -> pd.DataFrame:
    out = ROOT
    rows = []
    events, source = load_events(data, max(sizes(mode)), seed)
    for n in sizes(mode):
        graph, reps = graph_from(events.iloc[:n]), repetitions(n, mode)
        if graph.number_of_edges() == 0:
            continue
        degree = dict(graph.degree(weight="weight"))
        pr_time, pr_std, pr_mem = measure(lambda: pagerank(graph), reps)
        scores = pagerank(graph)
        top_pr = set(sorted(scores, key=scores.get, reverse=True)[:10])
        top_degree = set(sorted(degree, key=degree.get, reverse=True)[:10])
        record(rows, experiment="ranking", algorithm="PageRank", n=n, metric="solape_top10_grado",
               value=len(top_pr & top_degree) / 10, time_ms=pr_time, std_ms=pr_std, memory_mb=pr_mem,
               reps=reps, seed=seed, source=source)
        record(rows, experiment="ranking", algorithm="Grado ponderado", n=n, metric="solape_top10_grado",
               value=1, time_ms=0, std_ms=0, memory_mb=0, reps=reps, seed=seed, source=source)
        def louvain():
            return list(nx.community.louvain_communities(graph, weight="weight", seed=seed))
        l_time, l_std, l_mem = measure(louvain, reps)
        communities = louvain()
        mod = nx.community.modularity(graph, communities, weight="weight")
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="modularidad",
               value=mod, time_ms=l_time, std_ms=l_std, memory_mb=l_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="numero_comunidades",
               value=len(communities), time_ms=l_time, std_ms=l_std, memory_mb=l_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="tamano_mediano",
               value=float(sorted(map(len, communities))[len(communities) // 2]), time_ms=l_time, std_ms=l_std,
               memory_mb=l_mem, reps=reps, seed=seed, source=source)
        stable = partition_agreement(communities, list(nx.community.louvain_communities(graph, weight="weight", seed=seed + 1)), seed)
        record(rows, experiment="comunidades", algorithm="Louvain", n=n, metric="estabilidad_semilla",
               value=stable, time_ms=l_time, std_ms=l_std, memory_mb=l_mem, reps=reps, seed=seed, source=source)
        def lpa():
            return list(nx.community.asyn_lpa_communities(graph, weight="weight", seed=seed))
        p_time, p_std, p_mem = measure(lpa, reps)
        communities = lpa()
        mod = nx.community.modularity(graph, communities, weight="weight")
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="modularidad",
               value=mod, time_ms=p_time, std_ms=p_std, memory_mb=p_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="numero_comunidades",
               value=len(communities), time_ms=p_time, std_ms=p_std, memory_mb=p_mem, reps=reps, seed=seed, source=source)
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="tamano_mediano",
               value=float(sorted(map(len, communities))[len(communities) // 2]), time_ms=p_time, std_ms=p_std,
               memory_mb=p_mem, reps=reps, seed=seed, source=source)
        stable = partition_agreement(communities, list(nx.community.asyn_lpa_communities(graph, weight="weight", seed=seed + 1)), seed)
        record(rows, experiment="comunidades", algorithm="Label Propagation", n=n, metric="estabilidad_semilla",
               value=stable, time_ms=p_time, std_ms=p_std, memory_mb=p_mem, reps=reps, seed=seed, source=source)
    path = save(rows, "grafo", out)
    result = pd.read_csv(path)
    comunidades = result[(result["experimento"] == "comunidades") & (result["metrica"] == "modularidad")]
    line_chart(comunidades, "Comunidades: tiempo", "comunidades_tiempo.png", out)
    line_chart(comunidades, "Comunidades: modularidad", "comunidades_modularidad.png", out, value="valor", log=False)
    return result

resultado = run(mode=RUN_MODE, data=DATASET_FILE or None, seed=SEED)
resultado


## Lectura

El grado es rápido, pero PageRank distingue un producto puente de uno que solo se repite. Louvain se elige cuando su modularidad y su estabilidad entre semillas superan a Label Propagation.
